# 11 — Hyperbolic quadrant Step 3: \(\Delta W\) sign classification

Classify each weight update into four quadrants from `sign(ΔW1)` and `sign(ΔW2)` per epoch, plus a separate **zero** bucket when either increment is exactly 0.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
path_d1 = proc / "deltaW1_per_epoch.npy"
path_d2 = proc / "deltaW2_per_epoch.npy"

for p in (path_d1, path_d2):
    assert p.exists(), f"Missing {p} — run notebook 10 (Step 2) first."

deltaW1 = np.load(path_d1)
deltaW2 = np.load(path_d2)

assert deltaW1.shape == deltaW2.shape, f"Shape mismatch: {deltaW1.shape} vs {deltaW2.shape}"
n_epochs, n_visible, n_hidden = deltaW1.shape
total_elements = n_visible * n_hidden

print(f"deltaW1 shape: {deltaW1.shape}")
print(f"deltaW2 shape: {deltaW2.shape}")
print(f"elements per epoch: {total_elements:,}")

## Per-epoch quadrant counts

In [ ]:
LABELS = ["no_flip", "flip_both", "flip_delta2", "flip_delta1", "zero"]


def classify_epoch(d1, d2):
    """Return counts dict for one epoch's deltaW pair."""
    zero_mask = (d1 == 0.0) | (d2 == 0.0)
    s1 = np.sign(d1)
    s2 = np.sign(d2)

    active = ~zero_mask
    counts = {
        "no_flip": int(np.sum(active & (s1 > 0) & (s2 > 0))),
        "flip_both": int(np.sum(active & (s1 < 0) & (s2 < 0))),
        "flip_delta2": int(np.sum(active & (s1 > 0) & (s2 < 0))),
        "flip_delta1": int(np.sum(active & (s1 < 0) & (s2 > 0))),
        "zero": int(np.sum(zero_mask)),
    }
    counts["total"] = total_elements
    for k in LABELS:
        counts[f"p_{k}"] = counts[k] / total_elements
    counts["mixed_sign_ratio"] = (
        (counts["flip_delta1"] + counts["flip_delta2"]) / (total_elements - counts["zero"])
        if (total_elements - counts["zero"]) > 0
        else np.nan
    )
    return counts


rows = []
for epoch in range(n_epochs):
    c = classify_epoch(deltaW1[epoch], deltaW2[epoch])
    rows.append({"epoch": epoch, **c})

freq_df = pd.DataFrame(rows)
print(f"Built frequency table for {len(freq_df)} epochs.")

## Print sample epochs (every 5th)

In [ ]:
display_cols = ["epoch", "no_flip", "flip_both", "flip_delta2", "flip_delta1", "zero", "total"]
prop_cols = ["epoch"] + [f"p_{k}" for k in LABELS]

sample_epochs = list(range(0, n_epochs, 5))
if (n_epochs - 1) not in sample_epochs:
    sample_epochs.append(n_epochs - 1)

print("Counts (every 5th epoch):")
print(freq_df.loc[freq_df["epoch"].isin(sample_epochs), display_cols].to_string(index=False))

print("\nProportions (every 5th epoch):")
print(freq_df.loc[freq_df["epoch"].isin(sample_epochs), prop_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## Save and key metric

In [ ]:
out_csv = proc / "quadrant_frequency.csv"
freq_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

print("\n=== mixed_sign_ratio = (flip_delta1 + flip_delta2) / (total - zero) ===")
for ep in (0, 25, 49):
    if ep >= n_epochs:
        continue
    row = freq_df.loc[freq_df["epoch"] == ep].iloc[0]
    print(
        f"  epoch {ep:2d}: {row['mixed_sign_ratio']:.6f}  "
        f"(flip_delta1={int(row['flip_delta1']):,}, flip_delta2={int(row['flip_delta2']):,}, "
        f"non-zero={int(row['total'] - row['zero']):,})"
    )

msr = freq_df["mixed_sign_ratio"].values
print(f"\nAcross all epochs: min={msr.min():.6f}, max={msr.max():.6f}, mean={msr.mean():.6f}")
stable_positive = bool(np.all(msr > 0))
print(f"mixed_sign_ratio > 0 for all epochs: {stable_positive}")
if stable_positive:
    print("=> D-structure appears non-trivially active (opposite-sign updates persist).")
else:
    print("=> Some epochs have zero mixed-sign mass among non-zero updates.")